In [22]:
import pandas as pd

In [43]:
df1 = pd.read_csv(
    "UMSI/SIADS521/Data/UCMR5_All_Tribes_AK_LA.txt",
    sep="\t",
    dtype=str,
    encoding="cp1252"
)

df2 = pd.read_csv(
    "UMSI/SIADS521/Data/UCMR5_All_MA_WY.txt",
    sep="\t",
    dtype=str,
    encoding="cp1252"
)

df = pd.concat([df1, df2], ignore_index=True)

print("Rows loaded:", len(df))

Rows loaded: 1991977


In [44]:
df["MRL"] = pd.to_numeric(df["MRL"])
df["AnalyticalResultValue"] = pd.to_numeric(
    df["AnalyticalResultValue"]
)

In [45]:
df["AtOrAboveMRL"] = df["AnalyticalResultsSign"].eq("=")

In [46]:
df["AtOrAboveMRL"].value_counts()

AtOrAboveMRL
False    1934476
True       57501
Name: count, dtype: int64

In [47]:
states_50 = [
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA",
    "HI", "ID", "IL", "IN", "IA", "KS", "KY", "LA", "ME", "MD",
    "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ",
    "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC",
    "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY"
]

df_states = df[
    df["State"].isin(states_50)
].copy()

In [48]:
ref_contaminants = ["lithium", "PFBA", "PFHxA"]

df_states = df_states[
    df_states["Contaminant"].isin(ref_contaminants)
].copy()

In [49]:
df_states["CollectionDate"] = pd.to_datetime(
    df_states["CollectionDate"]
)

df_states = df_states[
    (df_states["CollectionDate"] >= "2023-01-01") &
    (df_states["CollectionDate"] < "2026-01-01")
].copy()

In [50]:
reference_concentrations = {
    "lithium": 10,
    "PFBA": 6,
    "PFHxA": 3
}

df_states["RefConc"] = df_states["Contaminant"].map(
    reference_concentrations
)

In [51]:
comparison_cutoffs = {
    "lithium": 15,
    "PFBA": 6.5,
    "PFHxA": 3.5
}

df_states["RefComparisonCutoff"] = df_states["Contaminant"].map(
    comparison_cutoffs
)

In [52]:
df_states["AboveRef"] = pd.Series(
    pd.NA,
    index=df_states.index,
    dtype="boolean"
)

has_reference = df_states["RefConc"].notna()

df_states.loc[has_reference, "AboveRef"] = (
    df_states.loc[has_reference, "AnalyticalResultValue"]
    > df_states.loc[has_reference, "RefComparisonCutoff"]
)

In [53]:
pws_summary = (
    df_states
    .groupby(["State", "PWSID", "Contaminant"])
    .agg(
        AtOrAboveMRL=("AtOrAboveMRL", "max"),
        AboveRef=("AboveRef", "max")
    )
    .reset_index()
)

In [54]:
state_summary = (
    pws_summary
    .groupby(["State", "Contaminant"])
    .agg(
        TotalPWS=("PWSID", "size"),
        AtOrAboveMRLPWS=("AtOrAboveMRL", "sum"),
        AboveRefPWS=("AboveRef", lambda x: x.sum(min_count=1))
    )
    .reset_index()
)

state_summary["AtOrAboveMRLPct"] = (
    state_summary["AtOrAboveMRLPWS"]
    / state_summary["TotalPWS"] * 100
)

state_summary["AboveRefPct"] = (
    state_summary["AboveRefPWS"]
    / state_summary["TotalPWS"] * 100
)

In [55]:
state_summary.to_csv(
    "UMSI/SIADS521/Data/state_summary.csv",
    index=False
)

In [56]:
value_data = df_states.loc[
    df_states["AtOrAboveMRL"],
    [
        "State",
        "PWSID",
        "Contaminant",
        "CollectionDate",
        "AnalyticalResultValue"
    ]
].copy()

In [57]:
value_data.to_csv(
    "UMSI/SIADS521/Data/value_data.csv",
    index=False
)

In [58]:
dates = pd.to_datetime(df["CollectionDate"])

measurement_data = df.loc[
    df["State"].isin(states_50) &
    (dates >= "2023-01-01") &
    (dates < "2026-01-01")
].copy()

In [59]:
measurement_data["AtOrAboveMRL"] = (
    measurement_data["AnalyticalResultsSign"].eq("=")
)

contaminant_summary = (
    measurement_data
    .groupby("Contaminant")
    .agg(
        TotalMeasurements=("AtOrAboveMRL", "size"),
        AtOrAboveMRLMeasurements=("AtOrAboveMRL", "sum")
    )
    .reset_index()
)

contaminant_summary["AtOrAboveMRLPct"] = (
    contaminant_summary["AtOrAboveMRLMeasurements"]
    / contaminant_summary["TotalMeasurements"] * 100
)

In [60]:
contaminant_summary.to_csv(
    "UMSI/SIADS521/Data/contaminant_summary.csv",
    index=False
)